In [0]:
# Import libraries
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
# Load master dataset
master_df = spark.read \
.option("header", True) \
.option("inferSchema", True) \
.csv("/Volumes/workspace/default/week7_data/customer_master.csv")

master_df.show()

+-----------+-------------+---------+---+--------------------+----------+
|customer_id|customer_name|     city|age|               email|     phone|
+-----------+-------------+---------+---+--------------------+----------+
|          1| Ananya Verma|   Jaipur| 35|customer1@example...|9362950628|
|          2|  Rahul Gupta|  Kolkata| 24|customer2@example...|9826600539|
|          3| Deepak Kumar|   Bhopal| 23|customer3@example...|9734036506|
|          4| Pooja Sharma|   Jaipur| 23|customer4@example...|9334760738|
|          5|Rahul Agarwal|  Chennai| 19|customer5@example...|9702632297|
|          6| Rohan Mishra|Hyderabad| 52|customer6@example...|9550455977|
|          7|  Rahul Mehta|  Chennai| 35|customer7@example...|9969119330|
|          8|   Aarav Soni|    Surat| 28|customer8@example...|9849621470|
|          9|  Pooja Patel|   Indore| 27|customer9@example...|9331191390|
|         10|Saurabh Patel|    Delhi| 23|customer10@exampl...|9507943839|
|         11| Vihaan Patel|   Nagpur| 

In [0]:
# Check schema
master_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: long (nullable = true)



In [0]:
# Count rows before cleaning
original_rows = master_df.count()
print("Rows Before Cleaning :", original_rows)

Rows Before Cleaning : 508


In [0]:
# Remove duplicate rows
master_df = master_df.dropDuplicates()

In [0]:
# Count after removing duplicates
print("Rows After Removing Duplicates :", master_df.count())

Rows After Removing Duplicates : 501


In [0]:
# Remove null values
master_df = master_df.na.drop()

In [0]:
# Count after cleaning
rows_in_master = master_df.count()
print("Rows After Cleaning (removing duplicates and dropping nulls) :",rows_in_master )
print("Reomved rows :", original_rows - rows_in_master)

Rows After Cleaning (removing duplicates and dropping nulls) : 488
Reomved rows : 20


In [0]:
# View cleaned data
master_df.show()

+-----------+--------------+-------+---+--------------------+----------+
|customer_id| customer_name|   city|age|               email|     phone|
+-----------+--------------+-------+---+--------------------+----------+
|         17|  Rahul Kapoor|  Delhi| 42|customer17@exampl...|9398471886|
|         32|   Aditya Soni| Jaipur| 25|customer32@exampl...|9264112119|
|         53|   Rahul Singh|  Delhi| 39|customer53@exampl...|9122585366|
|         86|  Ritika Singh| Bhopal| 34|customer86@exampl...|9242224154|
|         95|   Meera Mehta| Indore| 21|customer95@exampl...|9199104722|
|        102|    Arjun Jain| Mumbai| 20|customer102@examp...|9430989881|
|        122| Deepak Mishra|Kolkata| 60|customer122@examp...|9311564862|
|        133|    Ankit Soni|  Patna| 28|customer133@examp...|9807435606|
|        134|  Aditya Yadav| Bhopal| 60|customer134@examp...|9779615069|
|        141|   Aarav Kumar|  Surat| 24|customer141@examp...|9935822550|
|        150|  Ananya Singh| Indore| 46|customer150

In [0]:
# Create Delta table
master_df.write \
.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/week7_data/customer_delta")

In [0]:
# Read Delta table
delta_df = spark.read \
.format("delta") \
.load("/Volumes/workspace/default/week7_data/customer_delta")
delta_df.show()

+-----------+--------------+-------+---+--------------------+----------+
|customer_id| customer_name|   city|age|               email|     phone|
+-----------+--------------+-------+---+--------------------+----------+
|         17|  Rahul Kapoor|  Delhi| 42|customer17@exampl...|9398471886|
|         32|   Aditya Soni| Jaipur| 25|customer32@exampl...|9264112119|
|         53|   Rahul Singh|  Delhi| 39|customer53@exampl...|9122585366|
|         86|  Ritika Singh| Bhopal| 34|customer86@exampl...|9242224154|
|         95|   Meera Mehta| Indore| 21|customer95@exampl...|9199104722|
|        102|    Arjun Jain| Mumbai| 20|customer102@examp...|9430989881|
|        122| Deepak Mishra|Kolkata| 60|customer122@examp...|9311564862|
|        133|    Ankit Soni|  Patna| 28|customer133@examp...|9807435606|
|        134|  Aditya Yadav| Bhopal| 60|customer134@examp...|9779615069|
|        141|   Aarav Kumar|  Surat| 24|customer141@examp...|9935822550|
|        150|  Ananya Singh| Indore| 46|customer150

In [0]:
# Load incremental dataset
increment_df = spark.read \
.option("header", True) \
.option("inferSchema", True) \
.csv("/Volumes/workspace/default/week7_data/customer_incremental.csv")

increment_df.show()

+-----------+---------------+---------+---+--------------------+----------+
|customer_id|  customer_name|     city|age|               email|     phone|
+-----------+---------------+---------+---+--------------------+----------+
|        201|    Sneha Patel|Hyderabad| 35|customer201@examp...|9368018471|
|        202|  Vihaan Sharma|  Kolkata| 29|customer202@examp...|9635844374|
|        203|     Meera Jain|    Patna| 53|customer203@examp...|9226444910|
|        204|    Nitesh Soni|   Indore| 46|customer204@examp...|9330393232|
|        205|   Ishita Yadav|  Kolkata| 49|customer205@examp...|9314939857|
|        206|   Vihaan Gupta|   Nagpur| 22|customer206@examp...|9585377142|
|        207|  Krishna Kumar|  Kolkata| 46|customer207@examp...|9194264587|
|        208|    Riya Kapoor|Ahmedabad| 60|customer208@examp...|9473045064|
|        209|    Mohit Verma|   Bhopal| 52|customer209@examp...|9411774120|
|        210|    Aman Kapoor|   Mumbai| 58|customer210@examp...|9286928799|
|        211

In [0]:
# Clean incremental data
print("Rows Before Cleaning :", increment_df.count())
increment_df = increment_df.dropDuplicates()
increment_df = increment_df.na.drop()
rows_in_incremental = increment_df.count()
print("Rows After Cleaning :", rows_in_incremental)

Rows Before Cleaning : 200
Rows After Cleaning : 200


In [0]:
# Create DeltaTable object
# DeltaTable object because DataFrames are designed for querying and transformations, whereas Delta Lake operations such as MERGE, UPDATE, DELETE, and history() are available only through the DeltaTable API
deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/week7_data/customer_delta"
)

In [0]:
# Merge incremental data
merge_result  = deltaTable.alias("target") \
.merge(
    increment_df.alias("source"),
    "target.customer_id = source.customer_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
# Read final Delta table
final_df = spark.read \
.format("delta") \
.load("/Volumes/workspace/default/week7_data/customer_delta")

final_df.show()

+-----------+--------------+---------+---+--------------------+----------+
|customer_id| customer_name|     city|age|               email|     phone|
+-----------+--------------+---------+---+--------------------+----------+
|        273|   Pooja Gupta|    Noida| 31|customer273@examp...|9540639384|
|        290|   Mohit Yadav|   Mumbai| 24|customer290@examp...|9761078179|
|        502|    Aman Kumar|   Jaipur| 18|customer502@examp...|9420396197|
|        534| Nitesh Kapoor|   Jaipur| 31|customer534@examp...|9720956992|
|        541|   Arjun Joshi|   Nagpur| 40|customer541@examp...|9227381397|
|        559|    Neha Joshi|Hyderabad| 42|customer559@examp...|9943751263|
|        567|Aditya Agarwal|  Kolkata| 30|customer567@examp...|9155337035|
|        576| Karan Agarwal|Hyderabad| 40|customer576@examp...|9431165907|
|        587|   Deepak Soni|Ahmedabad| 28|customer587@examp...|9250279322|
|        595|  Ankit Sharma|   Jaipur| 52|customer595@examp...|9692322761|
|        212|   Arjun Sin

In [0]:
#Summary
final_rows = final_df.count()
new_rows = final_rows - rows_in_master
updated_rows = rows_in_incremental - new_rows
print("Rows in master        :", rows_in_master)
print("Rows in incremental   :", rows_in_incremental)
print("Final rows            :", final_rows)
print("New rows              :", new_rows)
print("Updated rows          :", updated_rows)
 # manual calculation

Rows in master        : 488
Rows in incremental   : 200
Final rows            : 592
New rows              : 104
Updated rows          : 96


In [0]:
merge_result.show() # automatically show above calculated merge_result

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|              200|              96|               0|              104|
+-----------------+----------------+----------------+-----------------+

